### Module 3: Conveyor Simulation  

In [7]:
import pandas as pd
import numpy as np

#### Clean Up the Input Data

In [8]:
# Read the raw CSVs
items_raw = pd.read_csv('Data/raw/order_itemtypes.csv', header=None)
qty_raw = pd.read_csv('Data/raw/order_quantities.csv', header=None)
tote_raw = pd.read_csv('Data/raw/orders_totes.csv', header=None)

# Add index column at the beginning
items_df = pd.concat([pd.Series(range(len(items_raw)), name='Index'), items_raw], axis=1)
qty_df = pd.concat([pd.Series(range(len(qty_raw)), name='Index'), qty_raw], axis=1)
tote_df = pd.concat([pd.Series(range(len(tote_raw)), name='Index'), tote_raw], axis=1)

# Create queues for orders and totes
from collections import deque
orders_queue = deque()
totes_queue = deque()
totes_dict = {}  # To group items by tote

# Process each order (row)
for order_num in range(len(items_df)):
    items_row = items_df.iloc[order_num]
    qty_row = qty_df.iloc[order_num]
    tote_row = tote_df.iloc[order_num]
    
    order_items = []
    
    # Process each column (skip column 0 which is now the index)
    for col_idx in range(1, len(items_row)):
        item = items_row.iloc[col_idx]
        qty = qty_row.iloc[col_idx]
        tote = tote_row.iloc[col_idx]
        
        # Skip empty cells (Weird that some totes are empty)
        if pd.isna(item) or str(item).strip() == '':
            continue
        
        item = int(float(item))
        qty = int(float(qty)) if pd.notna(qty) else 1
        tote = int(float(tote)) if pd.notna(tote) else 0
        
        # Add to order items
        order_items.append({'item': item, 'qty': qty, 'tote': tote})
        
        # Add to totes dictionary
        if tote not in totes_dict:
            totes_dict[tote] = []
        totes_dict[tote].append({'item': item, 'qty': qty})
    
    # Append order to queue if it has items
    if order_items:
        orders_queue.append({'order_num': order_num + 1, 'items': order_items})

# Create totes queue from totes dictionary
for tote_id in sorted(totes_dict.keys()):
    totes_queue.append({'tote_id': tote_id, 'items': totes_dict[tote_id]})

print("ORDERS QUEUE:")
print("=" * 70)
for order in orders_queue:
    print(f"Order {order['order_num']}:")
    for item_info in order['items']:
        print(f"  - {item_info['qty']} unit(s) of Item Type {item_info['item']} -> Tote {item_info['tote']}")

print("\n\nTOTES QUEUE:")
print("=" * 70)
for tote in totes_queue:
    print(f"Tote {tote['tote_id']}:")
    for item_info in tote['items']:
        print(f"  - {item_info['qty']} unit(s) of Item Type {item_info['item']}")


ORDERS QUEUE:
Order 1:
  - 3 unit(s) of Item Type 3 -> Tote 0
  - 2 unit(s) of Item Type 1 -> Tote 0
Order 2:
  - 1 unit(s) of Item Type 2 -> Tote 14
  - 3 unit(s) of Item Type 3 -> Tote 4
  - 1 unit(s) of Item Type 4 -> Tote 14
Order 3:
  - 2 unit(s) of Item Type 5 -> Tote 7
Order 4:
  - 3 unit(s) of Item Type 0 -> Tote 9
  - 1 unit(s) of Item Type 5 -> Tote 10
Order 5:
  - 1 unit(s) of Item Type 1 -> Tote 11
  - 1 unit(s) of Item Type 2 -> Tote 12
Order 6:
  - 2 unit(s) of Item Type 1 -> Tote 1
Order 7:
  - 1 unit(s) of Item Type 5 -> Tote 3
  - 2 unit(s) of Item Type 3 -> Tote 13
Order 8:
  - 2 unit(s) of Item Type 4 -> Tote 8
  - 1 unit(s) of Item Type 2 -> Tote 8
Order 9:
  - 1 unit(s) of Item Type 5 -> Tote 13
  - 3 unit(s) of Item Type 0 -> Tote 8
  - 2 unit(s) of Item Type 1 -> Tote 13
Order 10:
  - 3 unit(s) of Item Type 6 -> Tote 1
Order 11:
  - 1 unit(s) of Item Type 1 -> Tote 10


TOTES QUEUE:
Tote 0:
  - 3 unit(s) of Item Type 3
  - 2 unit(s) of Item Type 1
Tote 1:
  - 2 u

### Simulate the Conveyor Belts

In [ ]:
# Initialize conveyor system with assignments from solution_output.csv
from collections import deque

num_conveyors = 4
slots_per_conveyor = 2
conveyor_time = 12.0  # time units for item to traverse one conveyor (will modify later)
scan_interval = conveyor_time / 2  # Scan at each 0.5 second interval

# Create conveyors as lists with 2 slots each (None = empty)
# Conveyors are 1-based (1, 2, 3, 4) everywhere
conveyors = {i: [None, None] for i in range(1, num_conveyors + 1)}

# ============================================================
# READ SOLUTION OUTPUT FILE TO GET CONVEYOR ASSIGNMENTS
# ============================================================

# Load the solution file that assigns orders to conveyors
solution_df = pd.read_csv('Data/order_sequencing/algorithm1_load_balance_assignment.csv')
# solution_df = pd.read_csv('Data/order_sequencing/algorithm2_tote_overlap_assignment.csv')
# solution_df = pd.read_csv('Data/order_sequencing/algorithm3_itemtype_overlap_assignment.csv')
# solution_df = pd.read_csv('Data/order_sequencing/algorithm4_combined_overlap_assignment.csv')

print("SOLUTION FILE (Order -> Conveyor Assignments):")
print("=" * 70)
print(solution_df.to_string(index=False))

# Create a mapping of order_num -> conveyor_num from the solution file (1-based: 1-4)
order_assignment = {}  # order_num -> conveyor_num (1-4)
for idx, row in solution_df.iterrows():
    order_num = idx + 1  # Orders are 1-indexed
    conv_num = int(row['conv_num'])  # CSV uses 1-4; keep as-is
    order_assignment[order_num] = conv_num

print("\n\nOrder to Conveyor Mapping from Solution File:")
print("=" * 70)
for order_num in sorted(order_assignment.keys()):
    print(f"  Order {order_num} -> Conveyor {order_assignment[order_num]}")

# ============================================================
# BUILD ORDER QUEUES PER CONVEYOR
# ============================================================

# Create a working copy of orders_queue for reference
remaining_orders = list(orders_queue)

# Build a queue of orders for each conveyor (1-based: 1-4)
conveyor_order_queues = {i: deque() for i in range(1, num_conveyors + 1)}

# Group orders by their assigned conveyor
for order in remaining_orders:
    order_num = order['order_num']
    if order_num in order_assignment:
        conveyor_num = order_assignment[order_num]

        # Build items list (for display) and remaining as (item_type, tote_id) per unit (enforces tote assignment)
        items_list = []
        remaining_list = []  # each unit required: (item_type, tote_id)
        for item_info in order['items']:
            item = item_info['item']
            qty = int(item_info['qty'])
            tote_id = int(item_info['tote']) if item_info.get('tote') is not None else 0
            for _ in range(qty):
                items_list.append(item)
                remaining_list.append((item, tote_id))

        # Add to conveyor's queue
        conveyor_order_queues[conveyor_num].append({
            'order_num': order_num,
            'items': items_list.copy(),
            'remaining': remaining_list.copy(),
            'fulfilled': []
        })

# Print conveyor order queues
print("\nCONVEYOR ORDER QUEUES (based on solution_output.csv):")
print("=" * 70)
for conv_num in range(1, num_conveyors + 1):
    queue = conveyor_order_queues[conv_num]
    print(f"Conveyor {conv_num}: {len(queue)} orders")
    for order in queue:
        print(f"  - Order {order['order_num']}: {order['items']}")

# Current active order per conveyor (start with first order from each queue)
orders = {i: {'order_num': None, 'items': [], 'remaining': [], 'fulfilled': []}
          for i in range(1, num_conveyors + 1)}

def assign_next_order_to_conveyor(conveyor_num):
    """Assign the next order from the conveyor's queue"""
    if conveyor_order_queues[conveyor_num]:
        next_order = conveyor_order_queues[conveyor_num].popleft()
        orders[conveyor_num] = next_order
        return True
    else:
        orders[conveyor_num] = {'order_num': None, 'items': [], 'remaining': [], 'fulfilled': []}
        return False

# Assign first order to each conveyor
print("\nINITIAL ORDER ASSIGNMENTS:")
print("=" * 70)
for conv_num in range(1, num_conveyors + 1):
    if assign_next_order_to_conveyor(conv_num):
        print(f"Conveyor {conv_num} (Order {orders[conv_num]['order_num']}): {orders[conv_num]['items']}")
    else:
        print(f"Conveyor {conv_num}: No orders assigned")

# ============================================================
# TOTE SEQUENCING (switchable strategies)
# ============================================================

from collections import Counter

# Choose which tote sequencing strategy to use.
# - "fifo": always pick the smallest tote_id remaining
# - "smart_active_need": score totes by units needed by the 4 active orders (all conveyors equal)
# - "smart_active_need_weighted": same but weight conveyor 1 highest, 4 lowest (conveyor 1 sees items first)
TOTE_SEQ_ALGO = "smart_active_need"

# Weights for weighted method: conveyor 1 sees items first -> highest weight
CONVEYOR_WEIGHTS = {1: 4, 2: 3, 3: 2, 4: 1}

# Within-tote item order when loading a tote onto the belt.
# - "fixed": use natural order (item types as stored in tote)
# - "critical_first": put items that complete an order first, favor conveyor 1 over 2>3>4
WITHIN_TOTE_ORDER = "critical_first"

def _active_need_counts():
    """Counts of (item_type, tote_id) currently needed across the 4 active orders (enforces tote assignment)."""
    need = Counter()
    for c in range(1, num_conveyors + 1):
        if orders[c]['order_num'] is None:
            continue
        need.update(orders[c]['remaining'])  # remaining is list of (item_type, tote_id)
    return need


def _tote_score_smart_active_need(tote, need_counts):
    """Score = sum over (item, tote) of min(tote_qty, needed_qty for this tote). Need counts keyed by (item_type, tote_id)."""
    score = 0
    tid = tote['tote_id']
    for item_info in tote['items']:
        item = item_info['item']
        qty = int(item_info['qty'])
        score += min(qty, need_counts.get((item, tid), 0))
    return score


def _active_need_per_conveyor():
    """Per-conveyor counts of (item_type, tote_id) currently needed (for weighted scoring)."""
    need_per_conveyor = {}
    for c in range(1, num_conveyors + 1):
        if orders[c]['order_num'] is None:
            need_per_conveyor[c] = Counter()
        else:
            need_per_conveyor[c] = Counter(orders[c]['remaining'])
    return need_per_conveyor


def _tote_score_smart_active_need_weighted(tote, need_per_conveyor, weights):
    """Score = sum over conveyors c of weight[c] * (units in tote that c needs). Favors conveyor 1 (sees items first)."""
    score = 0
    tid = tote['tote_id']
    for item_info in tote['items']:
        item = item_info['item']
        qty = int(item_info['qty'])
        for c in range(1, num_conveyors + 1):
            need_c = need_per_conveyor.get(c, Counter()).get((item, tid), 0)
            score += weights.get(c, 1) * min(qty, need_c)
    return score


def select_next_tote(remaining_totes, algorithm):
    """Returns (tote_dict, score_used_for_selection)."""
    if not remaining_totes:
        return None, 0

    if algorithm == "fifo":
        tote = min(remaining_totes, key=lambda t: t['tote_id'])
        return tote, 0

    if algorithm == "smart_active_need":
        need_counts = _active_need_counts()
        best_tote = None
        best_score = -1

        for tote in remaining_totes:
            s = _tote_score_smart_active_need(tote, need_counts)
            if (s > best_score) or (
                s == best_score and (best_tote is None or tote['tote_id'] < best_tote['tote_id'])
            ):
                best_tote = tote
                best_score = s

        # If no tote helps active orders, fall back to FIFO.
        if best_score <= 0:
            tote = min(remaining_totes, key=lambda t: t['tote_id'])
            return tote, 0

        return best_tote, best_score

    if algorithm == "smart_active_need_weighted":
        need_per_conveyor = _active_need_per_conveyor()
        best_tote = None
        best_score = -1
        for tote in remaining_totes:
            s = _tote_score_smart_active_need_weighted(tote, need_per_conveyor, CONVEYOR_WEIGHTS)
            if (s > best_score) or (
                s == best_score and (best_tote is None or tote['tote_id'] < best_tote['tote_id'])
            ):
                best_tote = tote
                best_score = s
        if best_score <= 0:
            tote = min(remaining_totes, key=lambda t: t['tote_id'])
            return tote, 0
        return best_tote, best_score

    raise ValueError(f"Unknown TOTE_SEQ_ALGO: {algorithm}")


def _expand_tote_to_items(tote):
    """List of item types only (for display)."""
    expanded = []
    for item_info in tote['items']:
        item = item_info['item']
        qty = int(item_info['qty'])
        expanded.extend([item] * qty)
    return expanded


def _expand_tote_to_item_pairs(tote):
    """List of (item_type, tote_id) for each unit (for conveyor and order matching)."""
    pairs = []
    tid = tote['tote_id']
    for item_info in tote['items']:
        item = item_info['item']
        qty = int(item_info['qty'])
        pairs.extend([(item, tid)] * qty)
    return pairs


def _expand_tote_to_item_pairs_critical_first(tote, orders_ref, conveyor_weights):
    """Order units so critical items go out first: last item for an order first, then favor conveyor 1>2>3>4, then by remaining count."""
    pairs = []
    tid = tote['tote_id']
    for item_info in tote['items']:
        item = item_info['item']
        qty = int(item_info['qty'])
        pairs.extend([(item, tid)] * qty)
    # For each unit (item, tid), find best conveyor that needs it (smallest c = conveyor 1 first)
    def priority(unit):
        item, t = unit
        best_conv = None
        best_remaining = 999
        for c in range(1, num_conveyors + 1):
            if orders_ref[c]['order_num'] is None:
                continue
            rem = orders_ref[c]['remaining']
            if (item, t) not in rem:
                continue
            if best_conv is None or c < best_conv:
                best_conv = c
                best_remaining = len(rem)
        if best_conv is None:
            return 9999
        is_last = (best_remaining == 1)
        # Smaller = earlier: last items first (0), then by conveyor (1 first), then by remaining count
        return (0 if is_last else 1000) + best_conv * 10 + best_remaining
    pairs.sort(key=priority)
    return pairs


# Remaining totes in the system (each tote will be "loaded" once, as a batch)
remaining_totes = list(totes_queue)

# Items currently being streamed from the selected tote
current_tote_items = deque()

# Logs (for inspection/debugging)
loaded_totes = []
loaded_item_sequence = []

# Total items available across all totes
total_items = sum(int(info['qty']) for tote in remaining_totes for info in tote['items'])

print("\n\nTOTE SEQUENCING:")
print("=" * 70)
print(f"Tote sequencing algorithm: {TOTE_SEQ_ALGO}")
print(f"Within-tote item order: {WITHIN_TOTE_ORDER}")
print("Available totes (by tote_id; actual load order is chosen at each step by the algorithm above):")
for tote_info in sorted(remaining_totes, key=lambda t: t['tote_id']):
    tote_items = _expand_tote_to_items(tote_info)
    print(f"  Tote {tote_info['tote_id']}: {tote_items}")

# Track orders and fulfillment
time = 0
item_counter = 0
completed_orders_log = []  # Track when each order was completed

print("\n" + "=" * 70)
print(f"Total items to process: {total_items}")
print("\n" + "=" * 70)

def scan_and_remove_items():
    """
    Scan the second slot (index 1) of each conveyor.
    Each order is only fulfilled from its corresponding conveyor.
    When an order is complete, assign the next order from that conveyor's queue.
    """
    global orders
    items_removed = []
    orders_completed = []  # Track completed orders for later printing
    
    for conveyor_idx in range(1, num_conveyors + 1):
        slot_content = conveyors[conveyor_idx][1]  # (item_type, tote_id) or None

        # Skip if no item or no order assigned
        if slot_content is None or orders[conveyor_idx]['order_num'] is None:
            continue
        item_type, tote_id = slot_content

        # Only fulfill if this (item_type, tote_id) is required by this order (tote assignment)
        pair = (item_type, tote_id)
        if pair in orders[conveyor_idx]['remaining']:
            current_order_num = orders[conveyor_idx]['order_num']

            conveyors[conveyor_idx][1] = None
            orders[conveyor_idx]['remaining'].remove(pair)
            orders[conveyor_idx]['fulfilled'].append(pair)
            items_removed.append((conveyor_idx, item_type, current_order_num))

            # Check if order is complete (no remaining items)
            if len(orders[conveyor_idx]['remaining']) == 0:
                completed_order_num = current_order_num

                # Assign next order from this conveyor's queue
                if assign_next_order_to_conveyor(conveyor_idx):
                    orders_completed.append((conveyor_idx, completed_order_num,
                                           orders[conveyor_idx]['order_num'],
                                           orders[conveyor_idx]['remaining'].copy()))
                else:
                    orders_completed.append((conveyor_idx, completed_order_num, None, None))
    
    return items_removed, orders_completed

def simulate_conveyor_step(item_counter):
    """
    At each time step (conveyor_time/2 intervals):
    1. Move items from conveyor N slot 2 to conveyor N+1 slot 1 (reverse order to avoid conflicts)
    2. Move items from slot 1 to slot 2 within each conveyor (if slot 2 is empty)
       - But NOT for conveyors that just received items in step 1
    3. Add new item to Conveyor 1, slot 1 (load point). Conveyors are 1-4.
    """
    added_items = []
    conveyors_received_item = set()  # Track which conveyors just received items
    already_processed_slot2 = set()  # Track conveyors whose slot 2 was already processed

    # Step 1: Loop-back from Conveyor 4 to Conveyor 1 (if applicable)
    if conveyors[num_conveyors][1] is not None:
        loop_item = conveyors[num_conveyors][1]
        conveyors[num_conveyors][1] = None
        already_processed_slot2.add(num_conveyors)

        # Cascade slot 2 forward: 3->4, 2->3, 1->2
        for cascade_idx in range(num_conveyors - 1, 0, -1):
            if conveyors[cascade_idx][1] is not None:
                cascade_item = conveyors[cascade_idx][1]
                conveyors[cascade_idx][1] = None
                already_processed_slot2.add(cascade_idx)

                conveyors[cascade_idx + 1][1] = conveyors[cascade_idx + 1][0]
                conveyors[cascade_idx + 1][0] = cascade_item
                conveyors_received_item.add(cascade_idx + 1)
                added_items.append(f"Item {cascade_item[0]} moved from Conveyor {cascade_idx} slot 2 to Conveyor {cascade_idx + 1} slot 1")

        # Insert looped item at Conveyor 1
        conveyors[1][1] = conveyors[1][0]
        conveyors[1][0] = loop_item
        conveyors_received_item.add(1)
        added_items.append(f"Item {loop_item[0]} moved from Conveyor {num_conveyors} slot 2 back to Conveyor 1 slot 1 (LOOP)")

    else:
        # No loop: process slot 2 movements (4, 3, 2, 1)
        for conveyor_idx in range(num_conveyors, 0, -1):
            if conveyors[conveyor_idx][1] is not None:
                item = conveyors[conveyor_idx][1]
                conveyors[conveyor_idx][1] = None
                already_processed_slot2.add(conveyor_idx)

                if conveyor_idx < num_conveyors:
                    conveyors[conveyor_idx + 1][1] = conveyors[conveyor_idx + 1][0]
                    conveyors[conveyor_idx + 1][0] = item
                    conveyors_received_item.add(conveyor_idx + 1)
                    added_items.append(f"Item {item[0]} moved from Conveyor {conveyor_idx} slot 2 to Conveyor {conveyor_idx + 1} slot 1")

    # Step 2: Move slot 1 -> slot 2 within each conveyor
    for conveyor_idx in range(1, num_conveyors + 1):
        if conveyor_idx not in conveyors_received_item:
            if conveyors[conveyor_idx][0] is not None and conveyors[conveyor_idx][1] is None:
                item = conveyors[conveyor_idx][0]
                conveyors[conveyor_idx][1] = item
                conveyors[conveyor_idx][0] = None
                added_items.append(f"Item {item[0]} moved within Conveyor {conveyor_idx} from slot 1 to slot 2")

    # Step 3: Add new item to Conveyor 1, slot 1 (load point)
    global remaining_totes, current_tote_items, loaded_totes, loaded_item_sequence

    if item_counter < total_items and conveyors[1][0] is None:
        if not current_tote_items and remaining_totes:
            selected_tote, _score = select_next_tote(remaining_totes, TOTE_SEQ_ALGO)

            for i, t in enumerate(remaining_totes):
                if t['tote_id'] == selected_tote['tote_id']:
                    remaining_totes.pop(i)
                    break

            loaded_totes.append(selected_tote['tote_id'])
            if WITHIN_TOTE_ORDER == "critical_first":
                current_tote_items.extend(_expand_tote_to_item_pairs_critical_first(selected_tote, orders, CONVEYOR_WEIGHTS))
            else:
                current_tote_items.extend(_expand_tote_to_item_pairs(selected_tote))

        if current_tote_items:
            item_type, tote_id = current_tote_items.popleft()
            conveyors[1][0] = (item_type, tote_id)
            added_items.append(f"Item {item_type} ADDED to Conveyor 1 slot 1 (from Tote {tote_id})")
            loaded_item_sequence.append(item_type)
            item_counter += 1
    
    return item_counter, added_items


# Run simulation
print("Conveyor System Simulation with Predefined Order Assignments")
print("=" * 70)

for step in range(total_items * 8):
    item_counter, added_items = simulate_conveyor_step(item_counter)
    time += conveyor_time / 2
    
    # Scan and remove items AFTER movement
    items_removed, orders_completed = scan_and_remove_items()
    
    print("\n")
    print(f"Time {time:.1f}:")

    for c_id in range(1, num_conveyors + 1):
        s1 = conveyors[c_id][0]
        s2 = conveyors[c_id][1]
        slot1 = s1[0] if s1 is not None else ''
        slot2 = s2[0] if s2 is not None else ''
        print(f"  Conveyor {c_id}: [{slot1}, {slot2}]")

        
    # Print items added
    for msg in added_items:
        print(f"  >> {msg}")
    
    # Print items removed
    if items_removed:
        for conv_id, item_type, order_id in items_removed:
            print(f"  ** Item {item_type} REMOVED from Conveyor {conv_id} for Order {order_id}")
    
    # Print orders completed (after removals)
    for conv_id, completed_order, new_order, new_items in orders_completed:
        # Log the completed order with time
        completed_orders_log.append({'order_num': completed_order, 'time': time, 'conveyor': conv_id})
        if new_order is not None:
            print(f"  *** ORDER {completed_order} COMPLETE on Conveyor {conv_id}! New Order {new_order} assigned: {new_items}")
        else:
            print(f"  *** ORDER {completed_order} COMPLETE on Conveyor {conv_id}! No more orders in queue.")
    
    # Check if all items processed and all orders done
    all_queues_empty = all(len(conveyor_order_queues[c]) == 0 for c in range(1, num_conveyors + 1))
    all_orders_done = all(orders[c]['order_num'] is None or len(orders[c]['remaining']) == 0
                         for c in range(1, num_conveyors + 1))
    belt_empty = all(conveyors[c][0] is None and conveyors[c][1] is None for c in range(1, num_conveyors + 1))
    
    if item_counter >= total_items and belt_empty and all_queues_empty and all_orders_done:
        break

print("\n" + "=" * 70)
print("Tote Loading Summary:")
print(f"Loaded tote order (chosen by {TOTE_SEQ_ALGO} at each load opportunity): {loaded_totes}")
print(f"Loaded item sequence: {loaded_item_sequence}")

print("\n" + "=" * 70)
print("Order Fulfillment Summary:")
print(f"Total orders processed: {len(completed_orders_log)}")
print("\nCompleted Orders:")
for entry in sorted(completed_orders_log, key=lambda x: x['order_num']):
    print(f"  Order {entry['order_num']}: Completed at Time {entry['time']:.1f} on Conveyor {entry['conveyor']}")

# Calculate statistics
if completed_orders_log:
    total_time = max(entry['time'] for entry in completed_orders_log)
    avg_time = sum(entry['time'] for entry in completed_orders_log) / len(completed_orders_log)
    print(f"\nTotal time to complete all orders: {total_time:.1f}")
    print(f"Average completion time per order: {avg_time:.1f}")


SOLUTION FILE (Order -> Conveyor Assignments):
 conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1       3         2          0         0     0     1      0      0
        1       0         0          0         2     0     1      0      0
        1       0         1          1         0     0     0      0      0
        2       0         2          0         3     0     0      0      0
        2       0         0          1         0     2     0      0      0
        2       0         2          0         0     0     0      0      0
        3       0         0          1         3     1     0      0      0
        3       0         0          0         0     0     0      3      0
        3       0         1          0         0     0     0      0      0
        4       3         0          0         0     0     1      0      0
        4       0         0          0         0     0     2      0      0


Order to Conveyor Mapping from Solution File:
  Ord